# Baseline RAG — HSBC Pillar 3 (pages 4–27)

> ## ⚠️ THIS IS THE BASELINE (non-agentic) PIPELINE — DELIBERATELY NAIVE
>
> This notebook is the **control** we will measure against. It is intentionally
> simple and has **known structural weaknesses** (spelled out in the final cell):
>
> 1. **Naive chunking.** Fixed-size recursive character splitting that is blind to
>    table structure. It *will* cut tables apart, so a number can end up in a chunk
>    **without its reporting date, its unit ($m/$bn/%), or its row label**.
> 2. **No way to verify retrieval.** We embed, retrieve top-k, and trust it. There
>    is no relevance grading.
> 3. **No recovery.** If retrieval misses, there is no re-query, no rewrite, no
>    web fallback — the model answers from whatever came back.
> 4. **One-shot generation.** A single `gpt-4o-mini` call, no self-check.
>
> **Deliberately absent:** grading, routing, query rewriting, web search.
> Those belong to the later agentic phase, not here.

**Stack:** Gemini embeddings (`models/gemini-embedding-001`) · Chroma (persisted to
`chroma_db/`) · top-k = 4 · `gpt-4o-mini` @ temperature 0, grounded on retrieved
context only.

## 0 · Setup

In [1]:
import os, glob, shutil, textwrap
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY missing (see .env)"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing (see .env)"

# Config — these collection settings are the BASELINE reference; the later
# structure-aware store must reuse them so the comparison is apples-to-apples.
EXTRACTED_DIR   = "data/extracted"
PERSIST_DIR     = "chroma_db"
COLLECTION      = "pillar3_baseline"
DISTANCE        = "cosine"
EMBED_MODEL     = "models/gemini-embedding-001"
GEN_MODEL       = "gpt-4o-mini"
CHUNK_SIZE      = 800     # deliberately smaller than a full table -> tables get cut
CHUNK_OVERLAP   = 100
TOP_K           = 4
print("config loaded")

config loaded


## 1 · Load the extracted content (pages 4–27)

We load the per-page Markdown produced by `extract_tables.py`. Each page becomes a
single `Document` tagged with `source_page`. Note we feed the **rendered Markdown
tables** straight in — the naive splitter below has no idea they are tables.

In [2]:
from langchain_core.documents import Document

paths = sorted(glob.glob(f"{EXTRACTED_DIR}/page-*.md"))
docs = []
for p in paths:
    page = int(os.path.basename(p).split("-")[1].split(".")[0])
    text = open(p, encoding="utf-8").read()
    docs.append(Document(page_content=text, metadata={"source_page": page}))
print(f"loaded {len(docs)} page documents (pages {docs[0].metadata['source_page']}–{docs[-1].metadata['source_page']})")

loaded 24 page documents (pages 4–27)


## 2 · Naive chunking (intentionally structure-blind)

`RecursiveCharacterTextSplitter` at a fixed size. The only metadata that survives is
`source_page`. There is **no** notion of a table, caption, header row, unit row, or
reporting period — exactly the gap we want to expose.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],   # generic prose separators; table-unaware
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} pages -> {len(chunks)} naive chunks "
      f"(size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

# Show the breakage concretely: find a chunk that contains KM1 numbers but has lost
# the date header row, so the figures are stranded without their reporting period.
for c in chunks:
    if "132.6" in c.page_content and "31 Dec 2025" not in c.page_content:
        print("\n--- EXAMPLE: a chunk with figures but NO date header (p"
              f"{c.metadata['source_page']}) ---")
        print(textwrap.shorten(c.page_content.replace("\n"," "), 300))
        break

24 pages -> 210 naive chunks (size=800, overlap=100)

--- EXAMPLE: a chunk with figures but NO date header (p5) ---
Highlights CET1 capital and ratio Our common equity tier 1 (‘CET1’) capital was $132.6bn and our related structural foreign exchange RWAs. We expect to restore our ratio remained at 14.9%, unchanged from 31 December 2024. This CET1 capital ratio within our target range through a combination of [...]


## 3 · Embed with Gemini and persist to `chroma_db/`

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import time

embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)

def _open_existing():
    if not os.path.isdir(PERSIST_DIR):
        return None
    try:
        s = Chroma(collection_name=COLLECTION, embedding_function=embeddings,
                   persist_directory=PERSIST_DIR,
                   collection_metadata={"hnsw:space": DISTANCE})
        return s if s._collection.count() > 0 else None
    except Exception:
        return None

vectorstore = _open_existing()
if vectorstore is not None:
    # REUSE the already-built store as-is — no re-chunk, no re-embed, no rebuild.
    print(f"reusing existing store: {vectorstore._collection.count()} vectors "
          f"(collection='{COLLECTION}') — NOT re-embedding")
else:
    # Fresh build only when chroma_db/ is absent. Gemini's FREE embedding tier
    # allows ~100 requests/minute, so add chunks in throttled batches + 429-retry.
    vectorstore = Chroma(collection_name=COLLECTION, embedding_function=embeddings,
                         persist_directory=PERSIST_DIR,
                         collection_metadata={"hnsw:space": DISTANCE})
    EMBED_BATCH, PAUSE_S = 50, 61
    def add_with_retry(batch_docs, tries=5):
        for _ in range(tries):
            try:
                vectorstore.add_documents(batch_docs); return
            except Exception as e:
                if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                    print("  rate-limited; sleeping 60s ..."); time.sleep(60)
                else:
                    raise
        raise RuntimeError("embedding failed after retries")
    for i in range(0, len(chunks), EMBED_BATCH):
        add_with_retry(chunks[i:i + EMBED_BATCH])
        done = min(i + EMBED_BATCH, len(chunks))
        print(f"  embedded {done}/{len(chunks)} chunks")
        if done < len(chunks):
            time.sleep(PAUSE_S)
    print(f"persisted {vectorstore._collection.count()} vectors to {PERSIST_DIR}/ "
          f"(collection='{COLLECTION}', distance={DISTANCE})")

reusing existing store: 210 vectors (collection='pillar3_baseline') — NOT re-embedding


## 4 · Retriever (top-k = 4)

In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
hits = retriever.invoke("CET1 capital ratio at 31 December 2025")
print(f"retrieved {len(hits)} chunks for a sample query; source pages:",
      [h.metadata["source_page"] for h in hits])

retrieved 4 chunks for a sample query; source pages: [6, 5, 17, 17]


## 5 · Grounded generation (`gpt-4o-mini`, temperature 0)

A single LLM call, answering **only** from retrieved context. The prompt asks for the
unit and reporting date — but the model can only comply if the naive chunk actually
carried them. That is the whole point.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)

PROMPT = ChatPromptTemplate.from_template(
    "You are a banking regulatory-disclosure assistant. Answer the QUESTION using "
    "ONLY the CONTEXT extracted from HSBC's Pillar 3 disclosure.\n"
    "Rules:\n"
    "- Use only facts present in the CONTEXT. If the figure, its reporting date, or "
    "its unit is not in the CONTEXT, reply exactly: 'The retrieved context does not "
    "contain enough information to answer.'\n"
    "- When you state a figure, include its unit ($m, $bn, %, bps) and its reporting "
    "date exactly as shown.\n\n"
    "CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"
)

def format_context(hits):
    return "\n\n".join(
        f"[chunk {i+1} | page {h.metadata['source_page']}]\n{h.page_content}"
        for i, h in enumerate(hits)
    )

def answer(question, k=TOP_K, show_sources=True):
    hits = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(question)
    msg = PROMPT.format_messages(context=format_context(hits), question=question)
    resp = llm.invoke(msg).content
    if show_sources:
        print(f"Q: {question}")
        print(f"   retrieved pages: {[h.metadata['source_page'] for h in hits]}")
        print(f"A: {resp}\n")
    return resp

## 6 · Smoke test

In [7]:
_ = answer("What was the CET1 capital ratio at 31 December 2025?")
_ = answer("What were the RWAs for credit risk (excluding counterparty credit risk)?")

Q: What was the CET1 capital ratio at 31 December 2025?
   retrieved pages: [17, 17, 6, 5]
A: At 31 December 2025, the CET1 capital ratio was 14.9%.

Q: What were the RWAs for credit risk (excluding counterparty credit risk)?
   retrieved pages: [19, 19, 19, 20]
A: The retrieved context does not contain enough information to answer.



## 🔚 Known structural weaknesses of this baseline (read me)

This pipeline is **intentionally** limited. Recording the failure modes here so the
later phases have a clear target:

| Weakness | Why it bites this document |
|---|---|
| **Structure-blind chunking** | Fixed-size splits cut tables mid-grid. A retrieved chunk can contain `888,647` with **no header row**, so the model can't know it's *Total RWAs, $m, at 31 Dec 2025*. This is the root cause of the date/unit failures. |
| **No retrieval verification** | We never check whether the top-4 chunks actually contain the answer; an irrelevant-but-similar chunk is used as if authoritative. |
| **No recovery** | A retrieval miss is terminal — no query rewrite, no re-retrieval, no web fallback. |
| **One-shot, no self-check** | A single `gpt-4o-mini` call. If it misreads a stranded number's unit ($m vs $bn) there is nothing to catch it. |

**Distinct failure class to watch in the comparison:** an answer can get the *figure*
right but the *unit* wrong (e.g. reporting `888,647 $bn` instead of `$m`, or
conflating `$m` RWA cells with the `$bn` KM1 totals). That is a different bug from
retrieving the wrong number, and we will track it separately.

**Deliberately NOT here:** grading · routing · query rewriting · web search.

---
## 7 · Stress test — 10 questions on the NAIVE baseline

Runs 10 questions through the **existing** pipeline against the **already-built**
`chroma_db` collection `pillar3_baseline` (reused as-is above — no re-chunk, no
re-embed, no rebuild). Still **no grading / routing / rewriting / web search**.

- The model is shown **only the bare question** — never the gold answer.
- Gold answers + traps are kept separate, used **only for post-hoc scoring**.
- Scoring is heuristic (explicit token checks, documented in the scoring cell); the
  nuanced read lives in `docs/stress-test-findings.md`.

In [8]:
# Gold is stored here ONLY for scoring after the fact. It is never passed to the model.
# Each "nums"/"parts" inner list = acceptable synonyms for ONE required element.
QUESTIONS = [
 dict(id=1, q="What is HSBC's CET1 capital ratio as at 31 December 2025?",
      gold="14.9% (unchanged vs 31 Dec 2024)",
      trap="number-conflation (5 quarter ratios 14.9/14.5/14.6/14.7/14.9)",
      primary="14.9", units=["%"], nums=[["14.9"]], parts=[["14.9"]],
      trap_nums=["14.5","14.6","14.7"]),
 dict(id=2, q="What was HSBC's CET1 capital (the $bn figure, not the ratio) at 31 Dec 2025?",
      gold="$132.6bn", trap="returns the ratio, or a prior-period value",
      primary="132.6", units=["bn"], nums=[["132.6"]], parts=[["132.6"]],
      trap_nums=["14.9","124.9"]),
 dict(id=3, q="What is HSBC's leverage ratio, and did it rise or fall over 2025?",
      gold="5.3%, DOWN from 5.6% at 31 Dec 2024",
      trap="temporal ('down from' may yield 5.6 as current)",
      primary="5.3", units=["%"], nums=[["5.3"]],
      parts=[["5.3"],["down","fell","fall","decreas","lower","reduc"]],
      trap_nums=[]),
 dict(id=4, q="What is the combined RWA for market risk and counterparty credit risk at 31 Dec 2025?",
      gold="38,490 + 42,380 = 80,870 $m (= $80.9bn)",
      trap="aggregation (one row, or Total 888,647, or fabricated)",
      primary="38,490", units=["m","bn"], nums=[["80,870","80.9","80,870m"]],
      parts=[["80,870","80.9"]], trap_nums=["888,647"]),
 dict(id=5, q="Is HSBC's reported CET1 ratio on a transitional or end-point basis?",
      gold="Both identical — IFRS9 transitional ended 1 Jan 2025, CRR II grandfathering ended 28 Jun 2025",
      trap="basis/qualifier (ignores caveat, just restates a number)",
      primary="end-point", units=[], nums=[["same","identical","both","no difference","unchanged"]],
      parts=[["transitional"],["end-point","end point"]], trap_nums=[]),
 dict(id=6, q="What were HSBC's LCR and NSFR at 31 Dec 2025, and what is the HQLA amount?",
      gold="LCR 137%, NSFR 143%, average HQLA $702bn",
      trap="compound (drops one of the three parts)",
      primary="137", units=["%","bn"], nums=[["137"],["143"],["702"]],
      parts=[["137"],["143"],["702"]], trap_nums=[]),
 dict(id=7, q="What net CET1 impact did the Hang Seng Bank privatisation have, and when?",
      gold="net 110 bps in January 2026 (day-one ~120 bps, ~10 bps offset)",
      trap="conflates 110/120/10 bps",
      primary="110", units=["bps"], nums=[["110"]],
      parts=[["110"],["january 2026","2026"]], trap_nums=["120"]),
 dict(id=8, q="What total capital charge does HSBC apply as a percentage of RWAs, and under which regulation?",
      gold="8% of RWAs, set by Article 92(1) of CRR II",
      trap="cross-reference (8% without the Article)",
      primary="8%", units=["%"], nums=[["8%","8 %"]],
      parts=[["8%","8 %"],["article 92","92(1)","92 (1)"]], trap_nums=[]),
 dict(id=9, q="What is HSBC Holdings' total RWA, and in what currency?",
      gold="$888,647m (= $888.6bn), US dollars",
      trap="units (omits USD, or $bn without noting source cells are $m)",
      primary="888,647", units=["m","bn"], nums=[["888,647","888.6"]],
      parts=[["888,647","888.6"],["dollar","usd","us$"]], trap_nums=[]),
 dict(id=10, q="What is the operational risk RWA, and how much did it change from 2024?",
      gold="120,716 $m vs 106,472 $m = +14,244 $m (~+$14.2bn)",
      trap="aggregation/delta (one year, or miscomputes)",
      primary="120,716", units=["m","bn"], nums=[["120,716"],["106,472"],["14,244","14.2"]],
      parts=[["120,716"],["106,472"],["14,244","14.2"]], trap_nums=[]),
]
print(len(QUESTIONS), "questions loaded (gold hidden from the model)")

10 questions loaded (gold hidden from the model)


In [9]:
# Run each BARE question through the existing baseline pipeline.
REFUSAL = "does not contain enough information"
results = []
for item in QUESTIONS:
    hits = vectorstore.as_retriever(search_kwargs={"k": TOP_K}).invoke(item["q"])
    pages = [h.metadata["source_page"] for h in hits]
    ctx = format_context(hits)
    ans = llm.invoke(PROMPT.format_messages(context=ctx, question=item["q"])).content.strip()
    results.append(dict(id=item["id"], q=item["q"], answer=ans, pages=pages, ctx=ctx))
    print(f"Q{item['id']:>2} | pages {pages}\n   {ans}\n")

Q 1 | pages [17, 6, 16, 5]
   The retrieved context does not contain enough information to answer.

Q 2 | pages [16, 17, 6, 5]
   The retrieved context does not contain enough information to answer.

Q 3 | pages [22, 21, 22, 21]
   HSBC's leverage ratio was 5.3% at 31 December 2025, down from 5.6% at 31 December 2024. Therefore, it fell over 2025.

Q 4 | pages [20, 20, 19, 19]
   The retrieved context does not contain enough information to answer.

Q 5 | pages [7, 6, 7, 17]
   The retrieved context does not contain enough information to answer.

Q 6 | pages [5, 6, 23, 25]
   HSBC's LCR at 31 December 2025 was 143%, and the NSFR was also 143%. The HQLA amount was $702bn at 31 December 2025.

Q 7 | pages [7, 17, 17, 7]
   The net CET1 capital impact of the privatisation of Hang Seng Bank was 110 bps upon taking effect in January 2026.

Q 8 | pages [18, 6, 7, 6]
   HSBC applies a total capital charge of 20.5% of RWAs under the Basel III regulation as of 31 December 2025.

Q 9 | pages [11,

In [10]:
# Heuristic, transparent scoring (NOT a pipeline grader — post-hoc evaluation only).
def has_any(text, toks): return any(t.lower() in text.lower() for t in toks)
def has_group(text, groups): return all(has_any(text, g) for g in groups)

rows = []
for item, r in zip(QUESTIONS, results):
    ans, ctx = r["answer"], r["ctx"]
    answered = REFUSAL.lower() not in ans.lower()
    retrieved_has = item["primary"].lower() in ctx.lower()          # was the fact retrieved?
    numerically_ok = answered and has_group(ans, item["nums"])      # required figure(s) present
    complete = answered and has_group(ans, item["parts"])           # all parts present
    unit_ok = (not item["units"]) or (answered and has_any(ans, item["units"]))
    grounded = (not answered) or (numerically_ok and retrieved_has) # used a fact that was retrieved
    trap_fired = not (numerically_ok and complete and unit_ok)
    rows.append(dict(
        Q=item["id"], retrieved_has_fact="Y" if retrieved_has else "N",
        grounded="Y" if grounded else "N",
        numerically_correct="Y" if numerically_ok else "N",
        complete="Y" if complete else "N",
        unit_ok="Y" if unit_ok else "N",
        trap_fired="FIRED" if trap_fired else "ok",
    ))

import pandas as pd
score = pd.DataFrame(rows).set_index("Q")
print(score.to_string())
print("\nPASS (grounded & numerically_correct & complete & unit_ok):",
      sum((rw["grounded"]=="Y" and rw["numerically_correct"]=="Y"
           and rw["complete"]=="Y" and rw["unit_ok"]=="Y") for rw in rows), "/ 10")

   retrieved_has_fact grounded numerically_correct complete unit_ok trap_fired
Q                                                                             
1                   Y        Y                   N        N       N      FIRED
2                   Y        Y                   N        N       N      FIRED
3                   Y        Y                   Y        Y       Y         ok
4                   N        Y                   N        N       N      FIRED
5                   N        Y                   N        N       Y      FIRED
6                   Y        N                   N        N       Y      FIRED
7                   Y        Y                   Y        Y       Y         ok
8                   N        N                   N        N       Y      FIRED
9                   N        N                   N        N       Y      FIRED
10                  Y        N                   N        N       Y      FIRED

PASS (grounded & numerically_correct & complete & u

In [11]:
# Full table: question | baseline answer | gold | grounded? | numerically correct? | complete? | trap
import pandas as pd
table = []
for item, r, rw in zip(QUESTIONS, results, rows):
    table.append({
        "question": item["q"],
        "baseline answer": r["answer"],
        "gold": item["gold"],
        "grounded?": rw["grounded"],
        "numerically correct?": rw["numerically_correct"],
        "complete?": rw["complete"],
        "trap": ("FIRED — " + item["trap"]) if rw["trap_fired"]=="FIRED" else "avoided",
    })
full = pd.DataFrame(table)
pd.set_option("display.max_colwidth", 60)
print(full.to_string(index=False))
full  # rendered below as well

                                                                                      question                                                                                                       baseline answer                                                                                          gold grounded? numerically correct? complete?                                                                  trap
                                     What is HSBC's CET1 capital ratio as at 31 December 2025?                                                  The retrieved context does not contain enough information to answer.                                                              14.9% (unchanged vs 31 Dec 2024)         Y                    N         N FIRED — number-conflation (5 quarter ratios 14.9/14.5/14.6/14.7/14.9)
                  What was HSBC's CET1 capital (the $bn figure, not the ratio) at 31 Dec 2025?                                                  The retrieved contex

,question,baseline answer,gold,grounded?,numerically correct?,complete?,trap
0,What is HSBC's CET1 capital ratio as at 31 December 2025?,The retrieved context does not contain enough informatio...,14.9% (unchanged vs 31 Dec 2024),Y,N,N,FIRED — number-conflation (5 quarter ratios 14.9/14.5/14...
1,"What was HSBC's CET1 capital (the $bn figure, not the ra...",The retrieved context does not contain enough informatio...,$132.6bn,Y,N,N,"FIRED — returns the ratio, or a prior-period value"
2,"What is HSBC's leverage ratio, and did it rise or fall o...","HSBC's leverage ratio was 5.3% at 31 December 2025, down...","5.3%, DOWN from 5.6% at 31 Dec 2024",Y,Y,Y,avoided
3,What is the combined RWA for market risk and counterpart...,The retrieved context does not contain enough informatio...,"38,490 + 42,380 = 80,870 $m (= $80.9bn)",Y,N,N,"FIRED — aggregation (one row, or Total 888,647, or fabri..."
4,Is HSBC's reported CET1 ratio on a transitional or end-p...,The retrieved context does not contain enough informatio...,"Both identical — IFRS9 transitional ended 1 Jan 2025, CR...",Y,N,N,"FIRED — basis/qualifier (ignores caveat, just restates a..."
5,"What were HSBC's LCR and NSFR at 31 Dec 2025, and what i...","HSBC's LCR at 31 December 2025 was 143%, and the NSFR wa...","LCR 137%, NSFR 143%, average HQLA $702bn",N,N,N,FIRED — compound (drops one of the three parts)
6,What net CET1 impact did the Hang Seng Bank privatisatio...,The net CET1 capital impact of the privatisation of Hang...,"net 110 bps in January 2026 (day-one ~120 bps, ~10 bps o...",Y,Y,Y,avoided
7,What total capital charge does HSBC apply as a percentag...,HSBC applies a total capital charge of 20.5% of RWAs und...,"8% of RWAs, set by Article 92(1) of CRR II",N,N,N,FIRED — cross-reference (8% without the Article)
8,"What is HSBC Holdings' total RWA, and in what currency?","At 31 December 2025, HSBC Holdings' total RWAs were $22,...","$888,647m (= $888.6bn), US dollars",N,N,N,"FIRED — units (omits USD, or $bn without noting source c..."
9,"What is the operational risk RWA, and how much did it ch...","The operational risk RWA is $120,716m as of 31 December ...","120,716 $m vs 106,472 $m = +14,244 $m (~+$14.2bn)",N,N,N,"FIRED — aggregation/delta (one year, or miscomputes)"


### Reading the table
- **retrieved_has_fact = Y but a failure flag** ⇒ the right page *was* retrieved yet the
  answer still failed → this isolates a **chunking** problem (the figure arrived stripped
  of its row label / date / unit) from a pure **retrieval miss** (`retrieved_has_fact = N`).
- **unit_ok = N with numerically_correct = Y** is the *distinct* failure class to watch:
  right figure, wrong unit (e.g. RWA `$m` cells reported as `$bn`).

Narrative analysis and evidence: `docs/stress-test-findings.md`.